# ✌️ Notebook 1: The Dual-Write Problem

Many services do two things in one handler: **write to the database** and **publish an event** to a message bus. They are *not* in the same transaction, so any failure between them leaves the system inconsistent.


## 🛠️ Setup

Start Postgres and Adminer (browse the DB at http://localhost:8080):

```bash
cd 04-patterns/outbox-and-cdc
docker compose up -d
uv sync
```

Adminer login: server=`postgres`, user=`demo`, password=`demo`, database=`outbox_demo`.

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window.


## 🟥 BAD: write DB, then publish — fail in between

In [ ]:
import psycopg, random
DSN = 'host=localhost port=5432 user=demo password=demo dbname=outbox_demo'

with psycopg.connect(DSN, autocommit=True) as conn:
    conn.execute('DROP TABLE IF EXISTS orders')
    conn.execute('CREATE TABLE orders (id SERIAL PRIMARY KEY, item TEXT, total INTEGER)')

published = []  # pretend this is Kafka

def place_order(item, total, crash_after_db=False):
    with psycopg.connect(DSN) as conn:
        with conn.transaction():
            cur = conn.execute('INSERT INTO orders(item,total) VALUES (%s,%s) RETURNING id',
                               (item, total))
            order_id = cur.fetchone()[0]
        # transaction committed here ✅
        if crash_after_db:
            raise RuntimeError('💥 crash after DB commit, before publish')
        published.append({'order_id': order_id, 'item': item, 'total': total})
        return order_id

try:
    place_order('book', 25, crash_after_db=True)
except RuntimeError as e:
    print(e)

with psycopg.connect(DSN) as conn:
    rows = conn.execute('SELECT * FROM orders').fetchall()
print('orders in DB :', rows)
print('events published:', published)
print('💔 the DB has the order, but no event was published — downstream systems will never know')


👉 Next: store the event **inside the same transaction** as the data, in an `outbox` table. A separate publisher reads the table and ships events.